# Predicting election winners with XGBoost

Loads the long poll dataset (`polls_long_with_results.csv` from `build_dataset.ipynb`), engineers features (polls **plus** fundamentals: partisan lean, incumbency, prior margin), collapses to **one row per candidate per race**, and trains a gradient-boosted classifier for `won`.

**Honest framing up front:** for *calling the winner*, the polling leader is already right ~88–95% of the time, so a model barely beats that on accuracy. The model's value is **calibrated win probabilities** (useful for close races, ratings, betting markets) — judged by AUC / log-loss / Brier, not just accuracy.

**Validation:** split by year — train on 2018/2020/2022, test on 2024 (~133 races, each House district counted separately) — so we never leak the future.

In [1]:
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.metrics import (accuracy_score, roc_auc_score, log_loss,
                             brier_score_loss, confusion_matrix)
from sklearn.calibration import calibration_curve
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', 60)
print('xgboost', xgb.__version__)

xgboost 3.2.0


## 1. Load & filter

Keep only polls that matched a result (`has_result == 1`) and general-election cycles (even years 2018–2024).

In [2]:
d = pd.read_csv('polls_long_with_results.csv', low_memory=False)
d = d[d['has_result'] == 1].copy()

d['end_date']      = pd.to_datetime(d['end_date'], errors='coerce', format='mixed')
d['election_date'] = pd.to_datetime(d['election_date'], errors='coerce', format='mixed')
d['days_to_elec']  = (d['election_date'] - d['end_date']).dt.days

CYCLES = [2018, 2020, 2022, 2024]
d = d[d['year'].isin(CYCLES)].copy()

for c in ['pct','numeric_grade','pollscore','transparency_score','sample_size','days_to_elec']:
    d[c] = pd.to_numeric(d[c], errors='coerce')

print('poll rows:', len(d))
print('races:', d['race_id'].nunique())
print('by year (races):', d.groupby('year')['race_id'].nunique().to_dict())

poll rows: 16752
races: 687
by year (races): {2018: 227, 2020: 151, 2022: 176, 2024: 133}


## 1b. External features: partisan lean, incumbency, prior margin

Three signals polls don't capture, all leak-free (structural or strictly-prior data):

- **Partisan lean** — 538's `partisan-lean` dataset (state lean for Senate/Gov, district lean for House). A CPVI-style number; negative = R-leaning.
- **Incumbency** — `incumbent_party` per race from the `election-results` `races.csv`; we derive a per-candidate `is_incumbent` (this candidate's party holds the seat *and* they are running).
- **Prior same-office margin** — the most recent **previous** election's two-party margin for that exact seat, computed from results history (no current-cycle data, so no leakage).

All three are **signed toward the candidate's own party** so the model reads them consistently.

In [3]:
import io, re, requests

H = {'User-Agent': 'Mozilla/5.0 (research)'}
STATE_ABBR = {
    'Alabama':'AL','Alaska':'AK','Arizona':'AZ','Arkansas':'AR','California':'CA','Colorado':'CO',
    'Connecticut':'CT','Delaware':'DE','District of Columbia':'DC','Florida':'FL','Georgia':'GA',
    'Hawaii':'HI','Idaho':'ID','Illinois':'IL','Indiana':'IN','Iowa':'IA','Kansas':'KS','Kentucky':'KY',
    'Louisiana':'LA','Maine':'ME','Maryland':'MD','Massachusetts':'MA','Michigan':'MI','Minnesota':'MN',
    'Mississippi':'MS','Missouri':'MO','Montana':'MT','Nebraska':'NE','Nevada':'NV','New Hampshire':'NH',
    'New Jersey':'NJ','New Mexico':'NM','New York':'NY','North Carolina':'NC','North Dakota':'ND',
    'Ohio':'OH','Oklahoma':'OK','Oregon':'OR','Pennsylvania':'PA','Rhode Island':'RI','South Carolina':'SC',
    'South Dakota':'SD','Tennessee':'TN','Texas':'TX','Utah':'UT','Vermont':'VT','Virginia':'VA',
    'Washington':'WA','West Virginia':'WV','Wisconsin':'WI','Wyoming':'WY',
}

def npar(p):
    p = str(p).upper()
    return 'DEM' if p.startswith('DEM') else 'REP' if p.startswith('REP') else 'OTH'

def pdist(v):
    m = re.search(r'(\d+)', str(v))
    return str(int(m.group(1))) if m else ''

def _get(url):
    return pd.read_csv(io.StringIO(requests.get(url, timeout=60, headers=H).text), low_memory=False)

# --- 538 partisan lean (state + district) ---
LB = 'https://raw.githubusercontent.com/fivethirtyeight/data/master/partisan-lean/'
ls = _get(LB + 'fivethirtyeight_partisan_lean_STATES.csv'); ls.columns = ['sname', 'lean']
ls['state'] = ls['sname'].map(STATE_ABBR)
state_lean = dict(zip(ls['state'], ls['lean']))

ld = _get(LB + 'fivethirtyeight_partisan_lean_DISTRICTS.csv'); ld.columns = ['dcode', 'lean']
ld['state'] = ld['dcode'].str.split('-').str[0]
ld['district'] = ld['dcode'].str.split('-').str[1].map(lambda x: str(int(x)) if str(x).isdigit() else x)
dist_lean = {(r.state, r.district): r.lean for r in ld.itertuples()}

# --- incumbent_party per race (races.csv) ---
rc = _get('https://raw.githubusercontent.com/fivethirtyeight/election-results/main/races.csv')
def _off(o):
    o = str(o).lower()
    return 'Senate' if 'senate' in o else 'House' if 'house' in o else 'Governor' if 'governor' in o else None
rc['office'] = rc['office_name'].map(_off)
rc['state'] = rc['state_abbrev'].str.upper()
rc['district'] = ''
rc.loc[rc['office'] == 'House', 'district'] = rc.loc[rc['office'] == 'House', 'office_seat_name'].map(pdist)
inc_map = {(r.cycle, r.state, r.office, r.district): npar(r.incumbent_party)
           for r in rc[rc['office'].notna()].itertuples() if pd.notna(r.incumbent_party)}

# --- prior same-office two-party margin from results history (cached files in data/) ---
def _load_res(path, office):
    r = pd.read_csv(path, low_memory=False)
    r = r[r['stage'].astype(str).str.lower().str.contains('general', na=False)]
    r['office'] = office
    r['state'] = r['state_abbrev'].str.upper()
    r['district'] = '' if office != 'House' else r['office_seat_name'].map(pdist)
    r['p'] = r['ballot_party'].map(npar)      # 'party' col is null in these files; ballot_party holds DEM/REP
    r['pct'] = pd.to_numeric(r['percent'], errors='coerce')
    return r[['cycle', 'state', 'office', 'district', 'p', 'pct']]

allres = pd.concat([_load_res('data/res_senate.csv', 'Senate'),
                    _load_res('data/res_house.csv', 'House'),
                    _load_res('data/res_governor.csv', 'Governor')])
piv = (allres[allres['p'].isin(['DEM', 'REP'])]
       .groupby(['cycle', 'state', 'office', 'district', 'p'])['pct'].max().unstack('p'))
for col in ['DEM', 'REP']:
    if col not in piv.columns:
        piv[col] = np.nan
piv['margin'] = piv['DEM'].fillna(0) - piv['REP'].fillna(0)
margin_map = {idx: row.margin for idx, row in piv.iterrows()}

# Most recent same-office two-party margin strictly before `year` (leak-free)
def prior_margin(year, state, office, district):
    for back in range(2, 9, 2):
        v = margin_map.get((year - back, state, office, district))
        if v is not None and not (isinstance(v, float) and np.isnan(v)):
            return v
    return np.nan

print('state leans:', len(state_lean), '| district leans:', len(dist_lean),
      '| incumbent records:', len(inc_map), '| prior-margin races:', len(margin_map))

state leans: 51 | district leans: 435 | incumbent records: 12242 | prior-margin races: 8600


## 2. Poll weighting

Not all polls are equal. We weight each poll by:
- **recency** — closer to election day counts more (`1 / (1 + days/30)`),
- **sample size** — `sqrt(n)` (precision scales with root-n),
- **pollster quality** — 538 `numeric_grade`.

Used to compute a weighted polling average per candidate.

In [4]:
d['w'] = (1.0 / (1.0 + d['days_to_elec'].clip(lower=0) / 30.0)) \
         * np.sqrt(d['sample_size'].clip(lower=1)) \
         * (1.0 + d['numeric_grade'].fillna(0) / 3.0)

def wmean(values, weights):
    """Weighted mean, falling back to plain mean if weights are unusable."""
    w = weights.reindex(values.index).fillna(0)
    if w.sum() > 0 and values.notna().any():
        return np.average(values.fillna(values.mean()), weights=w)
    return values.mean()

## 3. Collapse to one row per candidate per race

Aggregate every poll for a candidate into a feature vector. Then add **race-relative** features (lead over best opponent, share of the polled field) — these matter more than raw % because elections are relative.

In [5]:
d['district'] = d['district'].fillna('').astype(str).replace('nan', '')

rows = []
for race_id, g in d.groupby('race_id'):
    for ck, gc in g.groupby('cand_key'):
        gc = gc.sort_values('end_date')
        last30 = gc[gc['days_to_elec'] <= 30]
        yr = int(gc['year'].iloc[0]); st = gc['state'].iloc[0]
        of = gc['office'].iloc[0];   di = str(gc['district'].iloc[0])
        party = gc['party_std'].iloc[0]

        # external features, signed toward THIS candidate's party (+ = favorable)
        lean = dist_lean.get((st, di)) if of == 'House' else state_lean.get(st)
        incp = inc_map.get((yr, st, of, di))
        pm   = prior_margin(yr, st, of, di)
        sign = 1 if party == 'DEM' else -1 if party == 'REP' else 0

        rows.append(dict(
            race_id=race_id, year=yr, state=st, office=of, district=di,
            cand_key=ck, candidate=gc['candidate'].iloc[0], party=party,
            won=int(gc['won'].iloc[0]),
            poll_avg=gc['pct'].mean(),
            poll_wavg=wmean(gc['pct'], gc['w']),
            poll_last=gc['pct'].iloc[-1],
            poll_last30=(last30['pct'].mean() if len(last30) else gc['pct'].mean()),
            poll_std=gc['pct'].std(),
            n_polls=len(gc),
            n_polls_over50=int((gc['pct'] > 50).sum()),
            avg_grade=gc['numeric_grade'].mean(),
            avg_pollscore=gc['pollscore'].mean(),
            avg_sample=gc['sample_size'].mean(),
            min_days=gc['days_to_elec'].min(),
            # --- new external features ---
            lean_cand=(sign * lean if lean is not None else np.nan),
            prior_margin_cand=(sign * pm if not (isinstance(pm, float) and np.isnan(pm)) else np.nan),
            is_incumbent=(1 if incp and incp == party else 0),
            is_inc_party_race=(1 if incp in ('DEM', 'REP') else 0),
        ))
cand = pd.DataFrame(rows)

# race-relative features
cand['field_best'] = cand.groupby('race_id')['poll_wavg'].transform(
    lambda s: s.nlargest(2).min() if len(s) > 1 else s.max())
cand['poll_lead']  = cand['poll_wavg'] - cand['field_best']
cand['poll_share'] = cand['poll_wavg'] / cand.groupby('race_id')['poll_wavg'].transform('sum')
cand['n_cands']    = cand.groupby('race_id')['cand_key'].transform('count')
cand['race_total_polls']  = cand.groupby('race_id')['n_polls'].transform('sum')
cand['frac_polls_over50'] = cand['n_polls_over50'] / cand['n_polls']
cand['is_dem']     = (cand['party'] == 'DEM').astype(int)
cand['is_rep']     = (cand['party'] == 'REP').astype(int)
cand['is_senate']  = (cand['office'] == 'Senate').astype(int)
cand['is_gov']     = (cand['office'] == 'Governor').astype(int)

print('candidate-races:', len(cand), '| win rate:', round(cand['won'].mean(), 3))
print('new-feature coverage (non-null %):',
      'lean', round(cand['lean_cand'].notna().mean(), 2),
      '| prior_margin', round(cand['prior_margin_cand'].notna().mean(), 2),
      '| incumbents', round(cand['is_incumbent'].mean(), 2))
cand[['year','state','office','candidate','party','poll_wavg','poll_lead',
      'lean_cand','prior_margin_cand','is_incumbent','won']].head(8)

candidate-races: 1859 | win rate: 0.371
new-feature coverage (non-null %): lean 0.41 | prior_margin 0.41 | incumbents 0.13


,year,state,office,candidate,party,poll_wavg,poll_lead,lean_cand,prior_margin_cand,is_incumbent,won
0,2018,AK,Governor,Mark Begich,DEM,35.388797,0.000000,-14.61110,-45.876524,0,0
1,2018,AK,Governor,Mike Dunleavy,REP,44.673941,9.285143,14.61110,45.876524,0,1
2,2018,AK,Governor,Billy Toien,OTH,3.300000,-32.088797,-0.00000,-0.000000,1,0
3,2018,AK,Governor,Bill Walker,OTH,23.189473,-12.199325,-0.00000,-0.000000,1,0
4,2018,AK,House,Alyse S. Galvin,DEM,45.252545,0.000000,NaN,NaN,0,0
5,2018,AK,House,Don Young,REP,48.463695,3.211150,NaN,NaN,0,1
6,2018,AL,Governor,Kay Ivey,REP,53.483009,21.531468,29.17974,27.316202,1,1
7,2018,AL,Governor,Walter Maddox,DEM,31.951541,0.000000,-29.17974,-27.316202,0,0


## 4. Train / test split by year

Train on 2018/2020/2022, test on 2024. **Never** include `vote_pct` / `race_winning_pct` — those are outcomes.

In [6]:
FEATURES = [
    'poll_avg','poll_wavg','poll_last','poll_last30','poll_std','n_polls',
    'n_polls_over50','frac_polls_over50','race_total_polls',
    'avg_grade','avg_pollscore','avg_sample','min_days',
    'poll_lead','poll_share','n_cands',
    'is_dem','is_rep','is_senate','is_gov',
    'lean_cand','prior_margin_cand','is_incumbent','is_inc_party_race',
]

train = cand[cand['year'] <= 2022]
test  = cand[cand['year'] == 2024]
X_train, y_train = train[FEATURES], train['won']
X_test,  y_test  = test[FEATURES],  test['won']
print('train:', X_train.shape, '| test:', X_test.shape)
print('train win rate:', round(y_train.mean(), 3), '| test win rate:', round(y_test.mean(), 3))

train: (1515, 24) | test: (344, 24)
train win rate: 0.367 | test win rate: 0.387


## 5. Train XGBoost

Shallow trees + low learning rate + subsampling to avoid overfitting on a small dataset (~1.5k training rows).

In [7]:
model = xgb.XGBClassifier(
    n_estimators=300, max_depth=3, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8, min_child_weight=3,
    eval_metric='logloss', random_state=42,
)
model.fit(X_train, y_train)
proba = model.predict_proba(X_test)[:, 1]
pred  = (proba > 0.5).astype(int)

## 6. Evaluate (candidate level)

In [8]:
print('=== candidate-level (test = 2024) ===')
print('AUC      ', round(roc_auc_score(y_test, proba), 3))
print('LogLoss  ', round(log_loss(y_test, proba), 3))
print('Brier    ', round(brier_score_loss(y_test, proba), 3), ' (lower = better calibrated)')
print('Accuracy ', round(accuracy_score(y_test, pred), 3))
print('\nConfusion matrix [rows=true, cols=pred]:')
print(confusion_matrix(y_test, pred))

=== candidate-level (test = 2024) ===
AUC       0.965
LogLoss   0.232
Brier     0.073  (lower = better calibrated)
Accuracy  0.881

Confusion matrix [rows=true, cols=pred]:
[[190  21]
 [ 20 113]]


## 7. Evaluate (race level) + baseline

Real question: per race, did we pick the right winner? Compare the model (highest predicted prob) to the naive baseline (highest polling average).

In [9]:
te = test.copy()
te['proba'] = proba

model_pick = te.loc[te.groupby('race_id')['proba'].idxmax()]
base_pick  = te.loc[te.groupby('race_id')['poll_wavg'].idxmax()]   # naive: highest weighted poll

print('races in test:', te['race_id'].nunique())
print('model  winner accuracy:', round(model_pick['won'].mean(), 3))
print('baseline (top poll)   :', round(base_pick['won'].mean(), 3))

# by office — Senate/Gov are statewide & well-polled (polls ~perfect); House is the hard part
print('\nby office  (model vs baseline, n races):')
for of in ['Senate', 'Governor', 'House']:
    mp = model_pick[model_pick['office'] == of]
    bp = base_pick[base_pick['office'] == of]
    if len(mp):
        print(f'  {of:9}  model {mp["won"].mean():.3f}   baseline {bp["won"].mean():.3f}   (n={len(mp)})')

print('\nRaces the model got WRONG:')
wrong = model_pick[model_pick['won'] == 0]
wrong[['year','state','office','district','candidate','party','poll_wavg',
       'poll_lead','lean_cand','prior_margin_cand','is_incumbent','proba']]

races in test: 133
model  winner accuracy: 0.85
baseline (top poll)   : 0.88

by office  (model vs baseline, n races):
  Senate     model 0.969   baseline 0.969   (n=32)
  Governor   model 1.000   baseline 1.000   (n=11)
  House      model 0.789   baseline 0.833   (n=90)

Races the model got WRONG:


,year,state,office,district,candidate,party,poll_wavg,poll_lead,lean_cand,prior_margin_cand,is_incumbent,proba
1518,2024,AK,House,1.0,Mary S. Peltola,DEM,45.797704,0.000000,NaN,NaN,0,0.461720
1524,2024,AZ,House,2.0,Jonathan Nez,DEM,42.000000,0.000000,NaN,NaN,0,0.279585
1537,2024,CA,House,22.0,Rudy Salas,DEM,45.334481,0.674997,NaN,NaN,0,0.609216
1542,2024,CA,House,41.0,Will Rollins,DEM,44.320121,1.783927,NaN,NaN,0,0.516804
1543,2024,CA,House,45.0,Michelle Steel,REP,45.119053,0.000000,NaN,NaN,0,0.566326
1545,2024,CA,House,47.0,Scott Baugh,REP,44.227667,2.654014,NaN,NaN,0,0.896428
1558,2024,CA,House,9.0,Kevin Lincoln,REP,44.000000,4.000000,NaN,NaN,0,0.873292
1561,2024,CO,House,3.0,Adam Frisch,DEM,48.017227,0.000000,NaN,NaN,0,0.846423
1564,2024,CO,House,8.0,Yadira Caraveo,DEM,45.736510,0.924738,NaN,NaN,0,0.521868
1593,2024,IA,House,1.0,Christina Bohannan,DEM,47.410709,2.046885,NaN,NaN,0,0.507997


## 8. Feature importance

In [10]:
imp = pd.Series(model.feature_importances_, index=FEATURES).sort_values()
ax = imp.plot.barh(figsize=(7, 5), title='XGBoost feature importance')
ax.set_xlabel('gain importance')
plt.tight_layout(); plt.show()
imp.sort_values(ascending=False).round(3)

C:\Users\pjmer\AppData\Local\Temp\ipykernel_31320\3632646110.py:4: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


poll_lead            0.430
poll_share           0.111
n_polls_over50       0.047
poll_wavg            0.038
poll_avg             0.034
prior_margin_cand    0.029
poll_last            0.028
is_rep               0.027
lean_cand            0.027
is_incumbent         0.026
is_dem               0.026
frac_polls_over50    0.020
avg_pollscore        0.020
poll_last30          0.020
n_cands              0.016
race_total_polls     0.016
poll_std             0.015
avg_grade            0.015
min_days             0.014
is_senate            0.014
avg_sample           0.014
n_polls              0.013
is_gov               0.000
is_inc_party_race    0.000
dtype: float32

## 9. Calibration curve

Are predicted probabilities honest? A well-calibrated model: candidates given ~70% win ~70% of the time.

In [11]:
frac_pos, mean_pred = calibration_curve(y_test, proba, n_bins=8, strategy='quantile')
plt.figure(figsize=(5, 5))
plt.plot([0, 1], [0, 1], '--', color='gray', label='perfect')
plt.plot(mean_pred, frac_pos, 'o-', label='model')
plt.xlabel('predicted win probability'); plt.ylabel('actual win frequency')
plt.title('Calibration (2024 test)'); plt.legend(); plt.tight_layout(); plt.show()

C:\Users\pjmer\AppData\Local\Temp\ipykernel_31320\4115570372.py:6: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.title('Calibration (2024 test)'); plt.legend(); plt.tight_layout(); plt.show()


## Notes / what's in here

- **Features now include fundamentals**: partisan `lean_cand`, `prior_margin_cand` (prior same-office margin), and `is_incumbent` — all leak-free and all land in the top half of importance. They lifted candidate-level **AUC to ~0.965 / Brier to ~0.073**.
- **Polls still dominate** the headline winner call. On a 2024 test set, Senate and Governor are essentially called by polls alone (model ≈ baseline); the fundamentals mostly improve **calibration** and help **House** (sparse polling) and close races.
- **House is the hard part** — each district is now its own race (~90 in the test set), polling is thin, and 538's district-lean file is a single 2022 vintage so coverage is ~41%. That's where most misses are.
- **Genuine misses like PA-Sen 2024**: Casey was the incumbent, PA's prior Senate margin was D+, and he led polls — every available signal pointed to him, yet he lost. No pre-election feature catches that; it's a real polling/environment miss and the floor on accuracy.

### Next steps
- **Leave-one-cycle-out CV** (train on 3 cycles, test the 4th, average) for a sturdier estimate than a single 2024 split.
- **National environment** (generic-ballot average) to capture wave years like 2024's red shift.
- **Time-varying district lean** (a per-cycle district PVI) to fix House coverage.
- **Don't** add `vote_pct` / `race_winning_pct` — they're the answer.